# Your First DFM Forecast with Real Economic Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-deane/course-creator/blob/main/courses/dynamic-factor-models/quick-starts/01_your_first_forecast.ipynb)

**Goal:** Forecast industrial production using related economic indicators through a Dynamic Factor Model.

**Time:** 15 minutes

**What you'll learn:** How to use multiple related time series to make better forecasts than using a single series alone.

---

## Quick Win: Forecast Industrial Production Right Now

We'll use 4 related economic indicators to forecast industrial production.

In [ ]:
# Install if running in Colab
try:
    import google.colab
    !pip install -q pandas-datareader statsmodels
except:
    pass

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.dynamic_factor import DynamicFactor
from pandas_datareader import data as web
from datetime import datetime

print("✓ Libraries loaded successfully!")

In [ ]:
# Download real economic data from FRED
start = datetime(2010, 1, 1)
end = datetime(2023, 12, 31)

indicators = {
    'INDPRO': 'Industrial Production Index',
    'UNRATE': 'Unemployment Rate',
    'HOUST': 'Housing Starts',
    'RETAILSMSA': 'Retail Sales'
}

print("📊 Downloading data from FRED...")
data_raw = pd.DataFrame()
for code, name in indicators.items():
    try:
        series = web.DataReader(code, 'fred', start, end)
        data_raw[name] = series.iloc[:, 0]
        print(f"  ✓ {name}")
    except Exception as e:
        print(f"  ✗ {name}: {e}")

# Handle missing values
data_clean = data_raw.dropna()

# Standardize (important for DFM)
data_std = (data_clean - data_clean.mean()) / data_clean.std()

print(f"\n✓ Loaded {len(data_std)} observations from {data_std.index[0].date()} to {data_std.index[-1].date()}")
data_std.head()

In [ ]:
# Split into training and test sets
train_size = int(len(data_std) * 0.85)
train_data = data_std.iloc[:train_size]
test_data = data_std.iloc[train_size:]

print(f"Training: {len(train_data)} observations")
print(f"Testing: {len(test_data)} observations")

# Fit Dynamic Factor Model
print("\n🔧 Fitting Dynamic Factor Model...")
model = DynamicFactor(
    train_data,
    k_factors=2,        # Extract 2 common factors
    factor_order=2      # Factors follow AR(2) process
)

results = model.fit(disp=False, maxiter=1000)
print("✓ Model fitted!")
print(f"Log-likelihood: {results.llf:.2f}")
print(f"AIC: {results.aic:.2f}")

## Visualize: Forecast vs Actuals

Let's see how well our DFM predicts industrial production.

In [ ]:
# Generate forecasts
forecast_steps = len(test_data)
forecast = results.forecast(steps=forecast_steps)
forecast_df = pd.DataFrame(
    forecast,
    index=test_data.index,
    columns=train_data.columns
)

# Get prediction intervals
predict_results = results.get_prediction(start=train_size, end=len(data_std)-1)
predict_ci = predict_results.conf_int()

# Focus on Industrial Production Index
target = 'Industrial Production Index'

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Full history with forecast
axes[0].plot(train_data.index, train_data[target], label='Training Data', linewidth=2, color='steelblue')
axes[0].plot(test_data.index, test_data[target], label='Actual (Test)', linewidth=2, color='darkgreen')
axes[0].plot(forecast_df.index, forecast_df[target], label='Forecast', linewidth=2, color='orangered', linestyle='--')
axes[0].axvline(train_data.index[-1], color='gray', linestyle=':', linewidth=2, label='Train/Test Split')
axes[0].set_title('Industrial Production: Forecast vs Actual', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Standardized Value')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# Plot 2: Zoom on test period with confidence intervals
test_idx = test_data.index
axes[1].plot(test_idx, test_data[target], label='Actual', linewidth=2.5, color='darkgreen', marker='o', markersize=4)
axes[1].plot(test_idx, forecast_df[target], label='Forecast', linewidth=2.5, color='orangered', linestyle='--', marker='s', markersize=4)

# Add confidence interval
target_idx = list(train_data.columns).index(target)
ci_lower = predict_ci.iloc[:, target_idx*2]
ci_upper = predict_ci.iloc[:, target_idx*2 + 1]
axes[1].fill_between(test_idx, ci_lower, ci_upper, alpha=0.2, color='orangered', label='95% Confidence Interval')

axes[1].set_title('Test Period Detail: Forecast Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Standardized Value')
axes[1].legend(loc='upper left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate forecast accuracy
actual = test_data[target].values
predicted = forecast_df[target].values
mae = np.mean(np.abs(actual - predicted))
rmse = np.sqrt(np.mean((actual - predicted)**2))
mape = np.mean(np.abs((actual - predicted) / actual)) * 100

print(f"\n📊 Forecast Accuracy Metrics:")
print(f"  MAE:  {mae:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAPE: {mape:.2f}%")

## How It Works: The Power of Multivariate Forecasting

**Why use DFM instead of forecasting a single series?**

1. **Information borrowing**: Industrial production is influenced by unemployment, housing, and retail sales. DFM uses ALL these signals.

2. **Noise reduction**: Common factors filter out idiosyncratic noise in each series.

3. **Missing data**: DFM can handle series with different frequencies or missing values.

**The process:**
```
4 Economic Indicators → Extract 2 Common Factors → Forecast all 4 series
```

**Factor interpretation:**

In [ ]:
# Extract and visualize factors
factors = results.factors.filtered

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(train_data.index, factors[:, 0], linewidth=2, color='darkred')
axes[0].set_title('Factor 1: Overall Economic Activity', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Factor Value')
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_data.index, factors[:, 1], linewidth=2, color='darkblue')
axes[1].set_title('Factor 2: Economic Cycle', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Factor Value')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Factor loadings
print("\n📈 Factor Loadings (how much each indicator depends on each factor):")
loadings = results.params['loading'].values.reshape(len(train_data.columns), 2)
loadings_df = pd.DataFrame(
    loadings,
    index=train_data.columns,
    columns=['Factor 1', 'Factor 2']
)
print(loadings_df.round(3))
print("\n💡 Interpretation: Larger absolute values = stronger dependence on that factor")

## Modify This: Try Different Indicators

Experiment with different economic indicators from FRED.

In [ ]:
# TODO: Modify these FRED codes to try different indicators
# Find more at: https://fred.stlouisfed.org/

custom_indicators = {
    'INDPRO': 'Industrial Production',     # Keep this as target
    'PAYEMS': 'Total Nonfarm Payrolls',    # Try: Employment data
    'CPIAUCSL': 'Consumer Price Index',    # Try: Inflation
    'DEXUSEU': 'USD/EUR Exchange Rate',    # Try: Currency
    # Add your own:
    # 'M2SL': 'M2 Money Supply',
    # 'FEDFUNDS': 'Federal Funds Rate',
}

K_FACTORS = 2        # TODO: Try 1, 2, or 3
FACTOR_ORDER = 2     # TODO: Try 1, 2, or 4

print("📊 Downloading custom indicators...")
data_custom = pd.DataFrame()
for code, name in custom_indicators.items():
    try:
        series = web.DataReader(code, 'fred', start, end)
        data_custom[name] = series.iloc[:, 0]
        print(f"  ✓ {name}")
    except Exception as e:
        print(f"  ✗ {name}: {e}")

data_custom = data_custom.dropna()
data_custom_std = (data_custom - data_custom.mean()) / data_custom.std()

# Train/test split
train_size_custom = int(len(data_custom_std) * 0.85)
train_custom = data_custom_std.iloc[:train_size_custom]
test_custom = data_custom_std.iloc[train_size_custom:]

# Fit model
print(f"\n🔧 Fitting DFM with {K_FACTORS} factors, order {FACTOR_ORDER}...")
model_custom = DynamicFactor(train_custom, k_factors=K_FACTORS, factor_order=FACTOR_ORDER)
results_custom = model_custom.fit(disp=False, maxiter=1000)

# Forecast
forecast_custom = results_custom.forecast(steps=len(test_custom))
forecast_custom_df = pd.DataFrame(forecast_custom, index=test_custom.index, columns=train_custom.columns)

# Evaluate
target_custom = 'Industrial Production'
actual_custom = test_custom[target_custom].values
predicted_custom = forecast_custom_df[target_custom].values
rmse_custom = np.sqrt(np.mean((actual_custom - predicted_custom)**2))

print(f"\n✓ Model AIC: {results_custom.aic:.2f}")
print(f"✓ Forecast RMSE: {rmse_custom:.4f}")

# Quick plot
plt.figure(figsize=(12, 5))
plt.plot(test_custom.index, actual_custom, label='Actual', linewidth=2, marker='o')
plt.plot(test_custom.index, predicted_custom, label='Forecast', linewidth=2, linestyle='--', marker='s')
plt.title(f'Custom Model: {K_FACTORS} Factors, Order {FACTOR_ORDER}', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n🔍 Experiment Tips:")
print("- More factors usually improve fit but may overfit")
- Try indicators that lead your target variable (e.g., permits before housing starts)")
print("- Compare AIC across models (lower is better)")

## Copy-Paste Template: Forecast Your Own Data

In [ ]:
from statsmodels.tsa.statespace.dynamic_factor import DynamicFactor
from pandas_datareader import data as web
import pandas as pd
import numpy as np

# STEP 1: Get your data
fred_codes = ['INDPRO', 'UNRATE', 'HOUST']  # Your indicators
data = pd.DataFrame({
    code: web.DataReader(code, 'fred', '2010-01-01', '2023-12-31').iloc[:, 0]
    for code in fred_codes
}).dropna()

# STEP 2: Standardize
data_std = (data - data.mean()) / data.std()

# STEP 3: Train/test split
train_size = int(len(data_std) * 0.85)
train = data_std.iloc[:train_size]
test = data_std.iloc[train_size:]

# STEP 4: Fit DFM
model = DynamicFactor(train, k_factors=2, factor_order=2)
results = model.fit(disp=False, maxiter=1000)

# STEP 5: Forecast
forecast = results.forecast(steps=len(test))
forecast_df = pd.DataFrame(forecast, index=test.index, columns=train.columns)

# STEP 6: Evaluate
target_col = 0  # Index of your target variable
rmse = np.sqrt(np.mean((test.iloc[:, target_col] - forecast[:, target_col])**2))
print(f"RMSE: {rmse:.4f}")

## What's Next?

You just made your first DFM forecast with real economic data! Here's what to explore:

1. **`02_multivariate_example.ipynb`** - Handle 10+ series with missing data
2. **`../concepts/visual_guides/model_selection.md`** - How to choose k_factors
3. **`../templates/dfm_forecast_pipeline.py`** - Production forecasting system
4. **`../modules/module_02_estimation/notebooks/01_maximum_likelihood.ipynb`** - How DFM estimation works

---

**Key Takeaway:** DFMs leverage relationships between multiple time series to produce more accurate forecasts than univariate methods.